In [2]:
 # Path to your CheXpert CSV

import os
import pandas as pd
import numpy as np
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

train_df_full = "/mnt/Internal/MedImage/merged_dataset.csv"  # update this to your file
train_df_full = pd.read_csv(train_df_full)
train_df_full = pd.get_dummies(train_df_full, columns=["GENDER", "PRIMARY_RACE", "ETHNICITY"])

train_df_full.columns.tolist()

# Convert categorical columns to integers in train_df_full

# Race

train_df_full['PRIMARY_RACE_American Indian or Alaska Native'] = train_df_full['PRIMARY_RACE_American Indian or Alaska Native'].astype(int)
train_df_full['PRIMARY_RACE_Asian'] = train_df_full['PRIMARY_RACE_Asian'].astype(int)
train_df_full['PRIMARY_RACE_Asian - Historical Conv'] = train_df_full['PRIMARY_RACE_Asian - Historical Conv'].astype(int)
train_df_full['PRIMARY_RACE_Asian, Hispanic'] = train_df_full['PRIMARY_RACE_Asian, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Asian, non-Hispanic'] = train_df_full['PRIMARY_RACE_Asian, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black or African American'] = train_df_full['PRIMARY_RACE_Black or African American'].astype(int)
train_df_full['PRIMARY_RACE_Black, Hispanic'] = train_df_full['PRIMARY_RACE_Black, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Black, non-Hispanic'] = train_df_full['PRIMARY_RACE_Black, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, Hispanic'] = train_df_full['PRIMARY_RACE_Native American, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native American, non-Hispanic'] = train_df_full['PRIMARY_RACE_Native American, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'] = train_df_full['PRIMARY_RACE_Native Hawaiian or Other Pacific Islander'].astype(int)
train_df_full['PRIMARY_RACE_Other'] = train_df_full['PRIMARY_RACE_Other'].astype(int)
train_df_full['PRIMARY_RACE_Other, Hispanic'] = train_df_full['PRIMARY_RACE_Other, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Other, non-Hispanic'] = train_df_full['PRIMARY_RACE_Other, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'] = train_df_full['PRIMARY_RACE_Pacific Islander, non-Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_Patient Refused'] = train_df_full['PRIMARY_RACE_Patient Refused'].astype(int)
train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'] = train_df_full['PRIMARY_RACE_Race and Ethnicity Unknown'].astype(int)
train_df_full['PRIMARY_RACE_Unknown'] = train_df_full['PRIMARY_RACE_Unknown'].astype(int)
train_df_full['PRIMARY_RACE_White'] = train_df_full['PRIMARY_RACE_White'].astype(int)
train_df_full['PRIMARY_RACE_White or Caucasian'] = train_df_full['PRIMARY_RACE_White or Caucasian'].astype(int)
train_df_full['PRIMARY_RACE_White, Hispanic'] = train_df_full['PRIMARY_RACE_White, Hispanic'].astype(int)
train_df_full['PRIMARY_RACE_White, non-Hispanic'] = train_df_full['PRIMARY_RACE_White, non-Hispanic'].astype(int)

# Ethnicity
#train_df_full['ETHNICITY_0'] = train_df_full['ETHNICITY_0'].astype(int)
train_df_full['ETHNICITY_Hispanic'] = train_df_full['ETHNICITY_Hispanic'].astype(int)
train_df_full['ETHNICITY_Hispanic/Latino'] = train_df_full['ETHNICITY_Hispanic/Latino'].astype(int)
train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'] = train_df_full['ETHNICITY_Non-Hispanic/Non-Latino'].astype(int)
train_df_full['ETHNICITY_Not Hispanic'] = train_df_full['ETHNICITY_Not Hispanic'].astype(int)
train_df_full['ETHNICITY_Patient Refused'] = train_df_full['ETHNICITY_Patient Refused'].astype(int)

# Gender
train_df_full['GENDER_Male'] = train_df_full['GENDER_Male'].astype(int)
train_df_full['GENDER_Female'] = train_df_full['GENDER_Female'].astype(int)


train_df_full.replace(-1, 1, inplace=True)

train_df_full.replace(np.nan, 0, inplace=True)


In [28]:
import pandas as pd
df = train_df_full # update this to your file
output_csv_path = '/mnt/Internal/MedImage/chexpert_balanced_for_training_10_per_label_dis+demog.csv'

# # Load data
df = df[0:175000]
print(df.columns.to_list)
# # The labels you're interested in
labels = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 
          'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'GENDER_Female', 'GENDER_Male',
          'PRIMARY_RACE_Asian','PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
          'PRIMARY_RACE_Asian, non-Hispanic', 'PRIMARY_RACE_Black or African American','PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
          'PRIMARY_RACE_White', 'PRIMARY_RACE_White or Caucasian',
          'PRIMARY_RACE_White, Hispanic', 'PRIMARY_RACE_White, non-Hispanic']
# # Optional: Convert -1 to 1 (treat uncertain as positive), or drop them
df[labels] = df[labels].replace(-1, 1)

# # Store selected rows
selected_rows = []

# # Keep track of used image paths to avoid duplicates
used_paths = set()

# # Select 300 images per label
for label in labels:
    subset = df[(df[label] == 1) & (~df['Path'].isin(used_paths))]  # Positive samples only and not used before
    selected = subset.sample(n=min(10, len(subset)), random_state=42)
    used_paths.update(selected['Path'].tolist())
    selected_rows.append(selected)

# # Combine all selected samples
final_df = pd.concat(selected_rows).drop_duplicates(subset='Path')
final_df.to_csv(output_csv_path, index=False)

print(f"Saved balanced subset to: {output_csv_path}")

<bound method IndexOpsMixin.tolist of Index(['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding',
       'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
       'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
       'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
       'Support Devices', 'PATIENT', 'AGE_AT_CXR', 'GENDER_Female',
       'GENDER_Male', 'GENDER_Unknown',
       'PRIMARY_RACE_American Indian or Alaska Native', 'PRIMARY_RACE_Asian',
       'PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
       'PRIMARY_RACE_Asian, non-Hispanic',
       'PRIMARY_RACE_Black or African American',
       'PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
       'PRIMARY_RACE_Native American, Hispanic',
       'PRIMARY_RACE_Native American, non-Hispanic',
       'PRIMARY_RACE_Native Hawaiian or Other Pacific Islander',
       'PRIMARY_RACE_Other', 'PRIMARY_RACE_Other, Hispanic',
       'PRIMAR

/tmp/ipykernel_844889/1601662466.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[labels] = df[labels].replace(-1, 1)


Saved balanced subset to: /mnt/Internal/MedImage/chexpert_balanced_for_training_10_per_label_dis+demog.csv


### Selecting Age also

In [3]:
import pandas as pd

# Load the dataset (update this if needed)
df = train_df_full
output_csv_path = '/mnt/Internal/MedImage/chexpert_balanced_for_training_3000_per_label_dis+demog+age.csv'

# Limit to a subset of the data
df = df[175000:]

# Create age group columns based on numeric Age
df['AGE_GROUP_AGE_0_30'] = ((df['Age'] >= 0) & (df['Age'] <= 30)).astype(int)
df['AGE_GROUP_AGE_31_50'] = ((df['Age'] >= 31) & (df['Age'] <= 50)).astype(int)
df['AGE_GROUP_AGE_51_70'] = ((df['Age'] >= 51) & (df['Age'] <= 70)).astype(int)
df['AGE_GROUP_AGE_71_plus'] = (df['Age'] >= 71).astype(int)

# Print column names for debugging
print(df.columns.to_list)

# The labels you're interested in
labels = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 
    'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 
    'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
    'GENDER_Female', 'GENDER_Male',
    'PRIMARY_RACE_Asian','PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
    'PRIMARY_RACE_Asian, non-Hispanic', 'PRIMARY_RACE_Black or African American','PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
    'PRIMARY_RACE_White', 'PRIMARY_RACE_White or Caucasian',
    'PRIMARY_RACE_White, Hispanic', 'PRIMARY_RACE_White, non-Hispanic',
    'AGE_GROUP_AGE_0_30', 'AGE_GROUP_AGE_31_50', 'AGE_GROUP_AGE_51_70', 'AGE_GROUP_AGE_71_plus'
]

# Optional: Convert -1 to 1 (treat uncertain as positive), or drop them
df[labels] = df[labels].replace(-1, 1)

# Store selected rows
selected_rows = []

# Keep track of used image paths to avoid duplicates
used_paths = set()

# Select 1000 images per label
for label in labels:
    subset = df[(df[label] == 1) & (~df['Path'].isin(used_paths))]  # Positive samples only and not used before
    selected = subset.sample(n=min(3000, len(subset)), random_state=42)
    used_paths.update(selected['Path'].tolist())
    selected_rows.append(selected)

# Combine all selected samples and remove duplicates
final_df = pd.concat(selected_rows).drop_duplicates(subset='Path')

# Save the final balanced subset
final_df.to_csv(output_csv_path, index=False)
print(f"Saved balanced subset to: {output_csv_path}")


/tmp/ipykernel_470002/2330501671.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AGE_GROUP_AGE_0_30'] = ((df['Age'] >= 0) & (df['Age'] <= 30)).astype(int)
/tmp/ipykernel_470002/2330501671.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AGE_GROUP_AGE_31_50'] = ((df['Age'] >= 31) & (df['Age'] <= 50)).astype(int)
/tmp/ipykernel_470002/2330501671.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

<bound method IndexOpsMixin.tolist of Index(['Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding',
       'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
       'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
       'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture',
       'Support Devices', 'PATIENT', 'AGE_AT_CXR', 'GENDER_Female',
       'GENDER_Male', 'GENDER_Unknown',
       'PRIMARY_RACE_American Indian or Alaska Native', 'PRIMARY_RACE_Asian',
       'PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
       'PRIMARY_RACE_Asian, non-Hispanic',
       'PRIMARY_RACE_Black or African American',
       'PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
       'PRIMARY_RACE_Native American, Hispanic',
       'PRIMARY_RACE_Native American, non-Hispanic',
       'PRIMARY_RACE_Native Hawaiian or Other Pacific Islander',
       'PRIMARY_RACE_Other', 'PRIMARY_RACE_Other, Hispanic',
       'PRIMAR